# Tamizaje sintético privado de Revisor-CRF

Esta libreta compara un modelo por sesión usando únicamente casos sintéticos. Configure en Colab Secrets un `GITHUB_TOKEN` fino, de sólo lectura para este repositorio y vigencia máxima de siete días. No escriba el token en una celda, no monte Drive y no cargue CRF, Excel, `results/` ni texto institucional.

In [ ]:
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True, capture_output=True, check=False)
if gpu.returncode != 0:
    raise RuntimeError("No hay GPU asignada. Seleccione un entorno GPU y reconecte.")
GPU_INFO = gpu.stdout.strip()
print(GPU_INFO)

## Descargar una copia limpia del repositorio privado

El token se lee desde Colab Secrets, se usa sólo en cabeceras HTTPS y se elimina de las variables de la libreta después de descargar el código.

In [ ]:
from google.colab import userdata
import io, json, os, pathlib, shutil, sys, tarfile, urllib.request
token = userdata.get("GITHUB_TOKEN")
if not token:
    raise RuntimeError("Falta GITHUB_TOKEN en Colab Secrets.")
headers = {"Authorization": f"Bearer {token}", "Accept": "application/vnd.github+json", "User-Agent": "revisor-crf-colab"}
commit_request = urllib.request.Request("https://api.github.com/repos/arodriguezg2106/PROYECTO-IA-ASF/commits/main", headers=headers)
with urllib.request.urlopen(commit_request, timeout=30) as response:
    SOURCE_COMMIT = json.load(response)["sha"]
archive_request = urllib.request.Request("https://api.github.com/repos/arodriguezg2106/PROYECTO-IA-ASF/tarball/main", headers=headers)
with urllib.request.urlopen(archive_request, timeout=120) as response:
    archive_bytes = response.read()
root = pathlib.Path("/content/revisor-source")
shutil.rmtree(root, ignore_errors=True)
root.mkdir()
with tarfile.open(fileobj=io.BytesIO(archive_bytes), mode="r:gz") as archive:
    archive.extractall(root, filter="data")
repo = next(path for path in root.iterdir() if path.is_dir())
del token, archive_bytes, headers
os.environ["REVISOR_SOURCE_COMMIT"] = SOURCE_COMMIT
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo)], check=True)
print("Commit evaluado:", SOURCE_COMMIT)

## Seleccionar e instalar un candidato


In [ ]:
MODEL_ID = "gemma3:12b" # @param ["gemma3:12b", "qwen3.5:9b"]
import time
if shutil.which("zstd") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "zstd"], check=True)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
log = open("/tmp/ollama.log", "w")
ollama_process = subprocess.Popen(["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT)
for _ in range(45):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama no inició; revise /tmp/ollama.log")
subprocess.run(["ollama", "pull", MODEL_ID], check=True)
warmup_payload = json.dumps({"model": MODEL_ID, "prompt": "Responde OK.", "stream": False, "think": False, "keep_alive": "10m", "options": {"num_predict": 2}}).encode("utf-8")
warmup_request = urllib.request.Request("http://127.0.0.1:11434/api/generate", data=warmup_payload, headers={"Content-Type": "application/json"})
with urllib.request.urlopen(warmup_request, timeout=300) as response:
    json.load(response)
print("Modelo descargado y precargado:", MODEL_ID)

## Ejecutar las compuertas 2 + 10 + 30

Cada etapa se ejecuta tres veces. Se guarda un checkpoint local al terminar cada compuerta y nunca se ejecuta la compuerta institucional.

In [ ]:
safe_model = MODEL_ID.replace(":", "-")
experiment_label = "prompt-v2"
output = pathlib.Path(f"/content/{safe_model}-{experiment_label}-synthetic.json")
checkpoint_dir = pathlib.Path(f"/content/{safe_model}-{experiment_label}-checkpoints")
environment = os.environ.copy()
environment["PYTHONPATH"] = f"{repo / 'src'}:{repo / 'scripts'}"
command = [sys.executable, str(repo / "scripts/evaluate_editorial_context.py"), "--synthetic-only", "--models", MODEL_ID, "--available-models", MODEL_ID, "--repetitions", "3", "--dataset", str(repo / "tests/data/editorial_context_cases.json"), "--config", str(repo / "editorial-config.example.toml"), "--checkpoint-dir", str(checkpoint_dir), "--output", str(output)]
started = time.perf_counter()
completed = subprocess.run(command, cwd=repo, env=environment, text=True)
elapsed = time.perf_counter() - started
if not output.exists():
    raise RuntimeError("La evaluación no generó el reporte.")
report = json.loads(output.read_text(encoding="utf-8"))
report["runtime"] = {"gpu": GPU_INFO, "ollama_version": subprocess.run(["ollama", "--version"], text=True, capture_output=True).stdout.strip(), "elapsed_seconds": round(elapsed, 3)}
output.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Código de salida:", completed.returncode, "Estado:", report["status"], "Modelo tamizado:", report["screened_model"], "Modelo aprobado:", report["selected_model"])

Descargue el JSON agregado y el ZIP de checkpoints. Repita la sesión con el otro modelo. Revoque el token al terminar.

In [ ]:
from google.colab import files
checkpoint_zip = shutil.make_archive(str(checkpoint_dir), "zip", checkpoint_dir)
files.download(str(output))
files.download(checkpoint_zip)